<a href="https://colab.research.google.com/github/2303A51689/Python-for-DS-1689/blob/main/new_dataset_with_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Supervised ML suite with classification reports
# Dataset: india_pesticide_toxicity_risk(_with_nulls).csv
# Tip: If auto-detection picks the wrong target, set TARGET_COLUMN manually.

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score

# Models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# ====== CONFIG ======
DATA_PATHS = [
    "india_pesticide_toxicity_risk_reduced.csv"
]
TARGET_COLUMN = None   # e.g., "toxicity_risk". Leave None to auto-detect.
TEST_SIZE = 0.2
RANDOM_STATE = 42

# ====== LOAD ======
for p in DATA_PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break
else:
    raise FileNotFoundError("Dataset not found. Please ensure the CSV is in /mnt/data/.")

# ====== TARGET AUTO-DETECTION ======
def auto_pick_target(dataframe):
    # Priority by name
    name_candidates = ["toxicity_risk","risk_label","label","target","class","toxicity","risk"]
    for c in dataframe.columns:
        if c.strip().lower() in name_candidates:
            return c
    # Otherwise: pick the last column if it's reasonably categorical or few unique values
    last = dataframe.columns[-1]
    # If last has too many unique numeric values, try to find a categorical-looking column
    if pd.api.types.is_numeric_dtype(dataframe[last]) and dataframe[last].nunique() > max(0.05*len(dataframe), 20):
        # pick a column with low cardinality
        low_card = [c for c in dataframe.columns if dataframe[c].nunique() <= max(0.05*len(dataframe), 20)]
        low_card = [c for c in low_card if c != last]
        if low_card:
            return low_card[-1]
    return last

y_col = TARGET_COLUMN or auto_pick_target(df)
if y_col not in df.columns:
    raise ValueError("Target column not found. Set TARGET_COLUMN to the correct label column.")

# ====== MAKE CLASSIFICATION TARGET IF NEEDED ======
y_raw = df[y_col]
# If numeric and high cardinality, discretize into 3 quantile bins (Low/Med/High)
if pd.api.types.is_numeric_dtype(y_raw) and y_raw.nunique() > max(0.05*len(df), 20):
    y = pd.qcut(y_raw, q=3, labels=["Low","Medium","High"])
else:
    y = y_raw.astype("category").astype(str)

X = df.drop(columns=[y_col])

# ====== FEATURE TYPES ======
numeric_features = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in X.columns if c not in numeric_features]

# ====== PREPROCESSORS ======
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ====== TRAIN/TEST SPLIT ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# ====== MODEL ZOO ======
models = {
    "LogisticRegression": LogisticRegression(max_iter=200, n_jobs=None if hasattr(LogisticRegression(), "n_jobs") else None, class_weight="balanced"),
    "LinearSVC": LinearSVC(),
    "SVC_RBF": SVC(kernel="rbf", probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "QuadraticDiscriminantAnalysis": QuadraticDiscriminantAnalysis(),
    "SGD_LogLoss": SGDClassifier(loss="log_loss", max_iter=1000, random_state=RANDOM_STATE)
}

# Some estimators (NB/LDA/QDA) need dense input; our OneHot is already dense.
# Build pipelines and evaluate
def fit_eval(name, clf):
    pipe = Pipeline(steps=[("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print("="*80)
    print(f"{name} | Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, digits=4))

for name, clf in models.items():
    try:
        fit_eval(name, clf)
    except Exception as e:
        print("="*80)
        print(f"{name} | SKIPPED due to error: {e}")

LogisticRegression | Accuracy: 0.5282
                  precision    recall  f1-score   support

IND-v0.1-modeled     0.9525    0.5312    0.6821      3810
             nan     0.0475    0.4684    0.0862       190

        accuracy                         0.5282      4000
       macro avg     0.5000    0.4998    0.3841      4000
    weighted avg     0.9095    0.5282    0.6538      4000

LinearSVC | Accuracy: 0.9525
                  precision    recall  f1-score   support

IND-v0.1-modeled     0.9525    1.0000    0.9757      3810
             nan     0.0000    0.0000    0.0000       190

        accuracy                         0.9525      4000
       macro avg     0.4763    0.5000    0.4878      4000
    weighted avg     0.9073    0.9525    0.9293      4000

SVC_RBF | Accuracy: 0.9525
                  precision    recall  f1-score   support

IND-v0.1-modeled     0.9525    1.0000    0.9757      3810
             nan     0.0000    0.0000    0.0000       190

        accuracy            

In [ ]:
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.optim as optim

# =============================
# Load dataset
# =============================
df = pd.read_csv("india_pesticide_toxicity_risk_reduced.csv")

# Fill nulls for simplicity
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(df[col].mean())

# Encode categorical variables
from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in df.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Normalize numeric
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[df.columns] = scaler.fit_transform(df[df.columns])

# =============================
# Define RL Environment
# =============================
class PesticideEnv:
    def __init__(self, data):
        self.data = data.reset_index(drop=True)
        self.n_features = data.shape[1]-1  # last col as target risk
        self.n_actions = 3  # Low / Medium / High pesticide use
        self.reset()

    def reset(self):
        self.state_index = np.random.randint(0, len(self.data))
        state = self.data.iloc[self.state_index, :-1].values
        return state

    def step(self, action):
        row = self.data.iloc[self.state_index]
        toxicity = row[-1]

        # Reward function
        if action == 0:  # Low
            reward = 1 - toxicity
        elif action == 1:  # Medium
            reward = 1 - abs(toxicity - 0.5)
        else:  # High
            reward = toxicity * -1

        done = True
        next_state = self.reset()
        return next_state, reward, done

# =============================
# Q-Learning
# =============================
def q_learning(env, episodes=500, alpha=0.1, gamma=0.9, epsilon=0.2):
    Q = {}
    rewards = []
    for ep in range(episodes):
        total_reward = 0
        state = tuple(env.reset())
        done = False
        while not done:
            if state not in Q:
                Q[state] = np.zeros(env.n_actions)
            if random.uniform(0,1) < epsilon:
                action = np.random.choice(env.n_actions)
            else:
                action = np.argmax(Q[state])
            next_state, reward, done = env.step(action)
            next_state = tuple(next_state)
            if next_state not in Q:
                Q[next_state] = np.zeros(env.n_actions)
            Q[state][action] += alpha * (reward + gamma * max(Q[next_state]) - Q[state][action])
            state = next_state
            total_reward += reward
        rewards.append(total_reward)
    return Q, rewards

# =============================
# SARSA
# =============================
def sarsa(env, episodes=500, alpha=0.1, gamma=0.9, epsilon=0.2):
    Q = {}
    rewards = []
    for ep in range(episodes):
        total_reward = 0
        state = tuple(env.reset())
        if state not in Q:
            Q[state] = np.zeros(env.n_actions)
        action = np.random.choice(env.n_actions)
        done = False
        while not done:
            next_state, reward, done = env.step(action)
            next_state = tuple(next_state)
            if next_state not in Q:
                Q[next_state] = np.zeros(env.n_actions)
            next_action = np.random.choice(env.n_actions) if random.uniform(0,1) < epsilon else np.argmax(Q[next_state])
            Q[state][action] += alpha * (reward + gamma * Q[next_state][next_action] - Q[state][action])
            state, action = next_state, next_action
            total_reward += reward
        rewards.append(total_reward)
    return Q, rewards

# =============================
# Deep Q-Network (DQN)
# =============================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, output_dim)
        )
    def forward(self, x):
        return self.fc(x)


def train_dqn(env, episodes=500):
    model = DQN(env.n_features, env.n_actions)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    rewards = []

    for ep in range(episodes):
        state = torch.FloatTensor(env.reset())
        total_reward = 0
        done = False
        while not done:
            q_values = model(state)
            action = torch.argmax(q_values).item() if random.uniform(0,1) > 0.1 else np.random.choice(env.n_actions)
            next_state, reward, done = env.step(action)
            next_state = torch.FloatTensor(next_state)
            reward_t = torch.tensor([reward], dtype=torch.float)

            target = q_values.clone().detach()
            target[action] = reward_t + 0.9 * torch.max(model(next_state)).detach()

            loss = loss_fn(q_values, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_reward += reward
            state = next_state
        rewards.append(total_reward)

    return model, rewards

# =============================
# Policy Gradient (REINFORCE)
# =============================
class PolicyNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(PolicyNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, output_dim), nn.Softmax(dim=-1)
        )
    def forward(self, x):
        return self.fc(x)


def train_reinforce(env, episodes=500):
    policy = PolicyNet(env.n_features, env.n_actions)
    optimizer = optim.Adam(policy.parameters(), lr=0.001)
    rewards = []

    for ep in range(episodes):
        state = torch.FloatTensor(env.reset())
        probs = policy(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        next_state, reward, done = env.step(action.item())
        loss = -dist.log_prob(action) * reward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        rewards.append(reward)
    return policy, rewards

# =============================
# Actor-Critic (A2C)
# =============================
class ActorCritic(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ActorCritic, self).__init__()
        self.actor = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, output_dim), nn.Softmax(dim=-1)
        )
        self.critic = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.actor(x), self.critic(x)


def train_a2c(env, episodes=500):
    model = ActorCritic(env.n_features, env.n_actions)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    rewards = []

    for ep in range(episodes):
        state = torch.FloatTensor(env.reset())
        probs, value = model(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        next_state, reward, done = env.step(action.item())
        next_state = torch.FloatTensor(next_state)
        _, next_value = model(next_state)

        advantage = reward + 0.9 * next_value.detach() - value
        actor_loss = -dist.log_prob(action) * advantage
        critic_loss = advantage.pow(2)

        loss = actor_loss + critic_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        rewards.append(reward)
    return model, rewards

# =============================
# Run and Get Outputs
# =============================
env = PesticideEnv(df)

Q_table, Q_rewards = q_learning(env, episodes=200)
print("Q-Learning average reward:", np.mean(Q_rewards))

SARSA_table, SARSA_rewards = sarsa(env, episodes=200)
print("SARSA average reward:", np.mean(SARSA_rewards))

DQN_model, DQN_rewards = train_dqn(env, episodes=200)
print("DQN average reward:", np.mean(DQN_rewards))

Policy_model, REINFORCE_rewards = train_reinforce(env, episodes=200)
print("REINFORCE average reward:", np.mean(REINFORCE_rewards))

A2C_model, A2C_rewards = train_a2c(env, episodes=200)
print("A2C average reward:", np.mean(A2C_rewards))

Q-Learning average reward: 0.855
SARSA average reward: 0.4775
DQN average reward: 0.4525
REINFORCE average reward: 0.655
A2C average reward: 0.7525


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# =============================
# Load and preprocess dataset
# =============================
df = pd.read_csv("india_pesticide_toxicity_risk_reduced.csv")

# Fill nulls
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(df[col].mean())

# Encode categoricals
encoders = {}
for col in df.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Normalize
scaler = MinMaxScaler()
df[df.columns] = scaler.fit_transform(df[df.columns])

# Assume last column is toxicity risk (target)
X = df.iloc[:, :-1].values
y = (df.iloc[:, -1].values > 0.5).astype(int)  # binary classification risk high/low

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

# =============================
# Feedforward Neural Network (ANN)
# =============================
class ANN(nn.Module):
    def __init__(self, input_dim):
        super(ANN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.fc(x)

# =============================
# CNN for Tabular Data
# =============================
class CNN(nn.Module):
    def __init__(self, input_dim):
        super(CNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool1d(4)
        )
        self.fc = nn.Sequential(
            nn.Linear(32*4, 32), nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# =============================
# LSTM (Temporal / Seasonal Analysis)
# =============================
class LSTMNet(nn.Module):
    def __init__(self, input_dim):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_dim, 64, batch_first=True)
        self.fc = nn.Linear(64, 2)
    def forward(self, x):
        x = x.unsqueeze(1)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

# =============================
# Autoencoder (Unsupervised Feature Learning)
# =============================
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, input_dim), nn.Sigmoid()
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

# =============================
# Training Utility
# =============================
def train_model(model, X_train, y_train, X_test, y_test, epochs=20):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = loss_fn(outputs, y_train)
        loss.backward()
        optimizer.step()
    preds = torch.argmax(model(X_test), dim=1).detach().numpy()
    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1-score:", f1_score(y_test, preds))
    print(classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

# =============================
# Run All Models
# =============================
input_dim = X_train.shape[1]

print("===== ANN =====")
ann = ANN(input_dim)
train_model(ann, X_train_t, y_train_t, X_test_t, y_test_t)

print("===== CNN =====")
cnn = CNN(input_dim)
train_model(cnn, X_train_t, y_train_t, X_test_t, y_test_t)

print("===== LSTM =====")
lstm = LSTMNet(input_dim)
train_model(lstm, X_train_t, y_train_t, X_test_t, y_test_t)

print("===== Autoencoder (unsupervised reconstruction) =====")
autoencoder = Autoencoder(input_dim)
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)
loss_fn = nn.MSELoss()
for ep in range(10):
    optimizer.zero_grad()
    recon, z = autoencoder(X_train_t)
    loss = loss_fn(recon, X_train_t)
    loss.backward()
    optimizer.step()
print("Autoencoder trained. Encoded feature dimension:", z.shape[1])


===== ANN =====
Accuracy: 0.95025
F1-score: 0.0
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3801
           1       0.00      0.00      0.00       199

    accuracy                           0.95      4000
   macro avg       0.48      0.50      0.49      4000
weighted avg       0.90      0.95      0.93      4000

Confusion Matrix:
 [[3801    0]
 [ 199    0]]
===== CNN =====
Accuracy: 0.95025
F1-score: 0.0
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3801
           1       0.00      0.00      0.00       199

    accuracy                           0.95      4000
   macro avg       0.48      0.50      0.49      4000
weighted avg       0.90      0.95      0.93      4000

Confusion Matrix:
 [[3801    0]
 [ 199    0]]
===== LSTM =====
Accuracy: 0.95025
F1-score: 0.0
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3801


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# =============================
# Load and preprocess dataset
# =============================
df = pd.read_csv("india_pesticide_toxicity_risk_reduced.csv")

# Fill nulls
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(df[col].mean())

# Encode categoricals
encoders = {}
for col in df.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Normalize
scaler = MinMaxScaler()
df[df.columns] = scaler.fit_transform(df[df.columns])

# Features & target
X = df.iloc[:, :-1].values
y = (df.iloc[:, -1].values > 0.5).astype(int)  # Binary classification risk high/low

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =============================
# Base Models
# =============================
log_reg = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
lgbm = LGBMClassifier()
cat = CatBoostClassifier(verbose=0)

# =============================
# Ensemble Techniques
# =============================
# Bagging -> Random Forest already included
# Boosting -> XGBoost, LightGBM, CatBoost

# Voting Classifier
voting = VotingClassifier(estimators=[
    ('lr', log_reg), ('rf', rf), ('xgb', xgb)
], voting='soft')

# Stacking Classifier
stacking = StackingClassifier(estimators=[
    ('dt', dt), ('rf', rf), ('xgb', xgb), ('lgbm', lgbm)
], final_estimator=LogisticRegression(max_iter=1000))

# =============================
# Train and Evaluate
# =============================
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"===== {name} =====")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1-score:", f1_score(y_test, preds))
    print(classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    print("\n")

# Evaluate base learners
evaluate_model("Logistic Regression", log_reg, X_train, y_train, X_test, y_test)
evaluate_model("Decision Tree", dt, X_train, y_train, X_test, y_test)
evaluate_model("Random Forest", rf, X_train, y_train, X_test, y_test)
evaluate_model("XGBoost", xgb, X_train, y_train, X_test, y_test)
evaluate_model("LightGBM", lgbm, X_train, y_train, X_test, y_test)
evaluate_model("CatBoost", cat, X_train, y_train, X_test, y_test)

# Evaluate ensembles
evaluate_model("Voting Ensemble", voting, X_train, y_train, X_test, y_test)
evaluate_model("Stacking Ensemble", stacking, X_train, y_train, X_test, y_test)


===== Logistic Regression =====
Accuracy: 0.95025
F1-score: 0.0
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3801
           1       0.00      0.00      0.00       199

    accuracy                           0.95      4000
   macro avg       0.48      0.50      0.49      4000
weighted avg       0.90      0.95      0.93      4000

Confusion Matrix:
 [[3801    0]
 [ 199    0]]


===== Decision Tree =====
Accuracy: 0.94225
F1-score: 0.008583690987124463
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      3801
           1       0.03      0.01      0.01       199

    accuracy                           0.94      4000
   macro avg       0.49      0.50      0.49      4000
weighted avg       0.90      0.94      0.92      4000

Confusion Matrix:
 [[3768   33]
 [ 198    1]]


===== Random Forest =====
Accuracy: 0.94575
F1-score: 0.0
              precision    recall  f1-score   support

In [ ]:
%pip install catboost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# =============================
# Load and preprocess dataset
# =============================
df = pd.read_csv("india_pesticide_toxicity_risk_reduced.csv")

# Fill nulls
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(df[col].mean())

# Encode categoricals
encoders = {}
for col in df.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Normalize
scaler = MinMaxScaler()
df[df.columns] = scaler.fit_transform(df[df.columns])

# Features & target
X = df.iloc[:, :-1].values
y = (df.iloc[:, -1].values > 0.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =============================
# Hyperparameter Tuning
# =============================

# CatBoost Tuning
cat_model = CatBoostClassifier(verbose=0, random_state=42)
cat_params = {
    'iterations': [200, 500],
    'depth': [6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5]
}
cat_search = RandomizedSearchCV(cat_model, cat_params, n_iter=5, scoring='f1', cv=3, random_state=42, n_jobs=-1)
cat_search.fit(X_train, y_train)
cat_best = cat_search.best_estimator_

# XGBoost Tuning
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_params = {
    'n_estimators': [200, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 1.0]
}
xgb_search = RandomizedSearchCV(xgb_model, xgb_params, n_iter=5, scoring='f1', cv=3, random_state=42, n_jobs=-1)
xgb_search.fit(X_train, y_train)
xgb_best = xgb_search.best_estimator_

# LightGBM Tuning
lgbm_model = LGBMClassifier(random_state=42)
lgbm_params = {
    'n_estimators': [200, 500],
    'max_depth': [-1, 6, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 50, 100]
}
lgbm_search = RandomizedSearchCV(lgbm_model, lgbm_params, n_iter=5, scoring='f1', cv=3, random_state=42, n_jobs=-1)
lgbm_search.fit(X_train, y_train)
lgbm_best = lgbm_search.best_estimator_

# =============================
# Stacking Ensemble with tuned models
# =============================
stacking = StackingClassifier(estimators=[
    ('xgb', xgb_best),
    ('lgbm', lgbm_best),
    ('cat', cat_best),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42))
], final_estimator=LogisticRegression(max_iter=1000), n_jobs=-1)

stacking.fit(X_train, y_train)

# =============================
# Evaluation Function
# =============================
def evaluate_model(name, model):
    preds = model.predict(X_test)
    print(f"===== {name} =====")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1-score:", f1_score(y_test, preds))
    print(classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    print("\n")

# Evaluate best tuned models
evaluate_model("Best CatBoost", cat_best)
evaluate_model("Best XGBoost", xgb_best)
evaluate_model("Best LightGBM", lgbm_best)
evaluate_model("Optimized Stacking Ensemble", stacking)

[LightGBM] [Info] Number of positive: 750, number of negative: 15250
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002866 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 468
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.046875 -> initscore=-3.012262
[LightGBM] [Info] Start training from score -3.012262
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
===== Best CatBoost =====
Accuracy: 0.94975
F1-score: 0.0
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3801
           1       0.00      0.00      0.00       199

    accuracy                           0.95      4000
   macro avg       0.48      0.50      0.49      4000
weighted avg       0.90      